# RAG demo for OpenIFS

## Project structure and layout

In [ ]:
from pathlib import Path

In [ ]:
# Root of this notebook
PROJECT_ROOT = Path("../").resolve()

# Data root
DATA_ROOT = Path(PROJECT_ROOT / "assets/data/rag_openifs").resolve()

# Where external repositories are cloned
DOCUMENTS_DIR = DATA_ROOT / "documents"

# Where FAISS indices and metadata are stored
INDEX_DIR = DATA_ROOT / "vector_db"

In [ ]:
# Create directories if they do not exist
DOCUMENTS_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)

In [ ]:
for name in [PROJECT_ROOT, DATA_ROOT, DOCUMENTS_DIR, INDEX_DIR]:
    print(f'{name.is_dir()!s:5} {name}')

## Ensuring example data are Git-ignored

In [ ]:
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

In [ ]:
ignore_entries = [
    "documents/",
    "vector_db/",
    "*.index",
    "*.faiss",
]

In [ ]:
for name in [GITIGNORE_PATH]:
    print(f'{name.exists()!s:5} {name}')

In [ ]:
# Read existing lines
if GITIGNORE_PATH.exists():
    gitignore_lines = GITIGNORE_PATH.read_text().split("\n")
else:
    gitignore_lines = []

# Check if we need to add anything
all_in_already = True
for new_entry in ignore_entries:
    if new_entry not in gitignore_lines:
        all_in_already = False
        break

# If so
if not all_in_already:

    # Get which lines
    new_lines = []
    if gitignore_lines[-1] != "\n":
        new_lines.append("")
    new_lines.append("# RAG example data")
    for new_entry in ignore_entries:
        if new_entry not in gitignore_lines:
            new_lines.append(new_entry)

    # Add them to file
    with GITIGNORE_PATH.open("a", encoding="utf-8") as fp:
        for new_line in new_lines:
            fp.write(new_line + "\n")
            print("Added to .gitignore:", new_line)

## Clone the OpenIFS model

In [ ]:
OPENIFS_REPO_URL = "https://github.com/ecmwf-ifs/openifs"

OPENIFS_DIR = DOCUMENTS_DIR / "openifs"

if not OPENIFS_DIR.is_dir():
    print("Cloning OpenIFS...")
    import subprocess
    subprocess.run(
        ["git", "clone", "--depth", "1", OPENIFS_REPO_URL, str(OPENIFS_DIR)],
        check = True
    )
    print("Done")
else:
    print("OpenIFS already exists - not cloning")

In [ ]:
# Sanity check - list top level
for item in sorted(OPENIFS_DIR.iterdir()):
    print(item)

## File selection rules

In [ ]:
# Skip the following
SKIP_DIRS = {
    ".git",
    "arch",
    "openifs-data",
}

In [ ]:
# Searchable extensions

# File extensions we consider searchable text/code
TEXT_EXTENSIONS = {
    ".f90", ".F90", ".F", ".intfb", ".fypp", ".func",
    ".c", ".h",
    ".py", ".sh", ".pl", ".ksh", ".bash",
    ".txt", ".md",
    ".cmake",
    ".yaml", ".yml",
    ".in", ".nam",
}

# Some important files have no extension
ALLOW_EXTENSIONLESS = {
    "LICENSE", "AUTHORS",
    "namelist", "namelists",
    "setup", "postprocessing",
    "ifs-run", "ifs-clean", "ifs-check-cvg", "ifs-check-tracers",
    "Dockerfile",
    "callscm",
    "NOTICE"
}

In [ ]:
# Skip large files for safety (2 MB)
MAX_FILE_SIZE = 2_000_000

In [ ]:
# File filter
def is_indexable_file(path: Path) -> bool:
    """Decide whether a file is indexable"""
    return (
        path.is_file() and
        all(part not in SKIP_DIRS for part in path.parts) and
        path.stat().st_size < MAX_FILE_SIZE and
        (
            path.suffix in TEXT_EXTENSIONS or
            path.suffix == "" and path.name in ALLOW_EXTENSIONLESS
        )
    )

In [ ]:
# Preview
indexable_files = []
all_files = []
excluded_files = []

for file in OPENIFS_DIR.rglob('*'):
    all_files.append(file)
    if is_indexable_file(file):
        indexable_files.append(file)
    else:
        excluded_files.append(file)

print("Indexable files:", len(indexable_files))
print("Excluded files: ", len(excluded_files))
print("Total files:    ", len(all_files), len(excluded_files)+len(indexable_files))
print()

for file in indexable_files[:10] + indexable_files[-10:]:
    print(file.relative_to(OPENIFS_DIR))

## Chunking

In [ ]:
# Text utilities

import re

def read_text_file(path: Path) -> str:
    """
    Read a text file robustly

    Try UTF-8 then fall back to latin-1
    """
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1", errors="replace")

def normalise_text(text: str) -> str:
    """
    Normalise white space and line endings

    Keep line structure (important for code)
    """
    # Line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Excessive blank lines
    text  = re.sub(r"\n{4,}", "\n\n\n", text)

    return text

In [ ]:
# Chunking function

from typing import List, Dict

def chunk_text_by_lines(
    text: str,
    target_chars: int = 1800,
    overlap_chars: int = 300,
) -> List[Dict]:
    """
    Split text into coverlapping chunks
    """
    
    lines = text.splitlines()
    chunks = []

    buffer = []
    buffer_len = 0
    start_line = 1

    for i, line in enumerate(lines):
        buffer.append(line)
        buffer_len += (len(line) + 1)

        if buffer_len >= target_chars:
            chunk_text = "\n".join(buffer).strip()
            end_line = i + 1

            chunks.append({
                "text": chunk_text,
                "line_start": start_line,
                "line_end": end_line,
            })

            # Prepare overlap
            overlap_buffer = []
            overlap_len = 0
            for prev_line in reversed(buffer):
                overlap_buffer.insert(0, prev_line)
                overlap_len += len(prev_line) + 1
                if overlap_len >= overlap_chars:
                    break

            buffer = overlap_buffer
            buffer_len = overlap_len
            start_line = end_line - len(buffer) + 1

    # Remaining text
    if buffer:
        chunks.append({
            "text": "\n".join(buffer).strip(),
            "line_start": start_line,
            "line_end": len(lines),
        })

    return chunks

In [ ]:
# Build OpenIFS chunks

skipped_files = 0
chunked_files = 0

chunks = []

for path in indexable_files:
    raw_text = read_text_file(path)
    
    # Skip trivial files
    if len(raw_text.strip()) < 100:
        skipped_files += 1
        continue

    text = normalise_text(raw_text)

    file_chunks = chunk_text_by_lines(text)

    for idx, chunk in enumerate(file_chunks):
        chunks.append({
            "file": str(path.relative_to(OPENIFS_DIR)),
            "chunk_id": idx,
            "line_start": chunk["line_start"],
            "line_end": chunk["line_end"],
            "text": chunk["text"],
        })

    chunked_files += 1

print(f"Total files chunked:     {chunked_files} out of {len(indexable_files)}")
print(f"Total files skipped:     {skipped_files} out of {len(indexable_files)}")
print(f"Total chunks created:    {len(chunks)}")
print(f"Average chunks per file: {(len(chunks) / len(indexable_files)):4.2f}")
print(f'Average chunk length:    {sum([len(chunk["text"]) for chunk in chunks]) / len(chunks):7.2f}')

In [ ]:
# Check a few chunks
import random

for idx in random.sample(range(0, len(chunks)), 2):
    print("\n\n> CHUNK", idx, "*"*120)
    print(f'> Lines  {chunks[idx]["line_start"]}-{chunks[idx]["line_end"]}')
    print(f'> File   {chunks[idx]["file"]}')
    print(f'> Length {len(chunks[idx]["text"])}')
    print("\n\n")
    print(chunks[idx]["text"][:500])

## Embedding

In [ ]:
# Load model

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

In [ ]:
model = SentenceTransformer(MODEL_NAME)

In [ ]:
embedding_dimension = model.get_embedding_dimension()

print("Loaded", MODEL_NAME)
print("Embedding dimension", embedding_dimension)

In [ ]:
texts = [chunk["text"] for chunk in chunks]

print(f"Total chunks created:    {len(chunks)}")
print(f"Total texts extracted:   {len(texts)}")

In [ ]:
# Compute embeddings

import numpy as np

embeddings = model.encode(
    texts,
    convert_to_numpy = True,
    show_progress_bar = True,
    batch_size = 32,
)

# Ensure FAISS-compatible datatype
embeddings = embeddings.astype(np.float32)

In [ ]:
print("Embedding array shape: ", embeddings.shape)

In [ ]:
# Sanity checks

assert embeddings.ndim == 2
assert embeddings.dtype == np.float32
assert embeddings.shape[0] == len(chunks)
assert embeddings.shape[1] == embedding_dimension

In [ ]:
# Norms

norms = np.linalg.norm(embeddings, axis=1)
print(  "min:", norms.min())
print(  "max:", norms.max())
print(  "mean:", norms.mean())

## Build FAISS index

In [ ]:
import faiss
faiss.omp_set_num_threads(1)   # avoid torch/faiss dual-libomp segfault on macOS

In [ ]:
index = faiss.IndexFlatIP(embedding_dimension)

In [ ]:
index.add(embeddings)

In [ ]:
print("FAISS index built")
print("Number of vectors: ", index.ntotal)

In [ ]:
# FAISS query function

def search_faiss(query: str, top_k: int=5):
    """
    Search the FAISS index for the most similar chunks
    Returns list of (score, metadata)
    """

    query_vector = model.encode(
        query,
        convert_to_numpy = True,
    ).astype("float32").reshape(1, -1)

    scores, indices = index.search(query_vector, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        metadata = chunks[idx]
        results.append((score, metadata))

    return results

In [ ]:
# Example query

query = "How is cloud microphysics treated in OpenIFS?"

results = search_faiss(query, top_k=3)

In [ ]:
for rank, (score, metadata) in enumerate(results):
    print("=" * 80)
    print(f'Rank {rank} | Similarity score {score:6.3f}')
    print(f'File {metadata["file"]}')
    print(f'Lines {metadata["line_start"]} - {metadata["line_end"]}')
    print("-" * 80)
    print(metadata["text"])

## Build context from retrieved chunks

In [ ]:
def build_context(query: str, top_k: int=8, max_chars: int=12000):
    """
    Retrieve top k chunks, build a context string that fits into max_chars
    """
    results = search_faiss(query, top_k=top_k)

    context_parts = []
    references = []
    total = 0

    for score, result in results:
        header = f'{result["file"]} | lines {result["line_start"]}-{result["line_end"]} | similarity={score:.3f}'
        body = result["text"].strip()
        block = header + "\n" + body

        block_len = len(block) + 2

        if total + block_len > max_chars:
            break

        context_parts.append(block)
        references.append((result["file"], result["line_start"], result["line_end"], float(score)))

        total += block_len

    context_text = "\n\n---\n\n".join(context_parts)
    return context_text, references

In [ ]:
%pip install ollama

In [ ]:
SYSTEM_PROMPT = """
You are a concise technical assistant for helping a user understand code.
Answer queries based strictly on the provided context.
Say if the context is insufficient.
Always end your answer with a 'References' section that gives sources.
"""

OLLAMA_MODEL="llama3.2:latest"

def answer_query(query: str, system_prompt: str, top_k: int=8):

    import ollama

    context_text, references = build_context(query, top_k=top_k)

    user_prompt = context_text + "\n\n---\n\n" + query
    
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "system", "content": system_prompt},
                 {"role": "user",   "content": user_prompt}
                ],
    )
    
    return response["message"]["content"], references

In [ ]:
answer, references = answer_query("How is cloud microphysics treated in OpenIFS?", SYSTEM_PROMPT)

In [ ]:
from IPython.display import display, Markdown

display(Markdown(answer))

display(Markdown("#### References:\n" +
"\n".join(["- " + ", ".join([str(el) for el in tpl]) for tpl in references])
))

In [ ]:
# The answer is bad; check the context:
context_text, references = build_context(query, top_k=3)
for ref in references:
    print(ref)

In [ ]:
# Widen the search
results = search_faiss(query, top_k=20)
for score, metadata in results:
    print(metadata["file"], score)    

In [ ]:
# Try with better embedding model
import ollama

batch_size = 1000

start = 0
end = 0
while end < len(texts):
    end = start + batch_size - 1
    batch = texts[start:end]
    print(len(batch))
    start += batch_size
 
#response = ollama.embed(model="nomic-embed-text", input=texts[0:1000])
#print(
#    type(response["embeddings"]),
#    len(response["embeddings"]),
#)
#embeddings = embeddings.astype(np.float32)